In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression

In [ ]:
import os

model_dir = "model" if os.path.exists("model") else os.path.join("..", "model")
src_dir = "src" if os.path.exists("src") else os.path.join("..", "src")
os.makedirs(model_dir, exist_ok=True)
os.makedirs(src_dir, exist_ok=True)

In [ ]:
import os
import pandas as pd
import numpy as np


candidate_paths = [
    "student_data_cleaned.csv",
    "StudentPerformanceFactors.csv",
    "student_data.csv",
    os.path.join("data", "student_data_cleaned.csv"),
    os.path.join("..", "data", "student_data_cleaned.csv")
]

data_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if not data_path:
    raise FileNotFoundError("Clean dataset not found! Please ensure 'student_data_cleaned.csv' is available.")

df = pd.read_csv(data_path)

In [ ]:
for col in ['Teacher_Quality', 'Parental_Education_Level', 'Distance_from_Home']:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])
df.loc[df['Exam_Score'] > 100, 'Exam_Score'] = 100

X = df.drop(columns=['Exam_Score'])
y = df['Exam_Score']

num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
    ],
    remainder='drop'
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_processed = preprocessor.fit_transform(X_train)

In [ ]:
from sklearn.linear_model import LinearRegression

production_model = LinearRegression()
production_model.fit(X_train_processed, y_train)

LinearRegression()

In [ ]:
import os
import joblib

joblib.dump(preprocessor, os.path.join(model_dir, "preprocessor.joblib"))
joblib.dump(production_model, os.path.join(model_dir, "best_student_model.joblib"))

print("✓ Saved pipeline artifacts:")
print(f"  - Preprocessor: {os.path.join(model_dir, 'preprocessor.joblib')}")
print(f"  - Model:        {os.path.join(model_dir, 'best_student_model.joblib')}")

✓ Saved pipeline artifacts:
  - Preprocessor: ../model/preprocessor.joblib
  - Model:        ../model/best_student_model.joblib


In [ ]:
def predict_student_performance(student_dict, model_path=None, preprocessor_path=None):
    """
    Accepts raw student features dictionary, applies transformations,
    and returns predicted Exam_Score bound to [0, 100].
    """
    if model_path is None:
        model_path = os.path.join(model_dir, "best_student_model.joblib")
    if preprocessor_path is None:
        preprocessor_path = os.path.join(model_dir, "preprocessor.joblib")

    model = joblib.load(model_path)
    prep = joblib.load(preprocessor_path)

    # Convert input dict to single-row DataFrame
    input_df = pd.DataFrame([student_dict])

    # Transform raw features using saved preprocessor
    features_processed = prep.transform(input_df)

    # Generate prediction
    raw_pred = model.predict(features_processed)[0]

    # Clip bounds within academic range [0, 100]
    final_score = float(np.clip(raw_pred, 0, 100))
    return round(final_score, 2)

print("✓ Inference function 'predict_student_performance()' defined and ready.")

✓ Inference function 'predict_student_performance()' defined and ready.


In [ ]:
# Sample Student Profiles
student_high = {
    'Hours_Studied': 28,
    'Attendance': 95,
    'Sleep_Hours': 7,
    'Previous_Scores': 90,
    'Tutoring_Sessions': 3,
    'Physical_Activity': 4,
    'Parental_Involvement': 'High',
    'Access_to_Resources': 'High',
    'Extracurricular_Activities': 'Yes',
    'Motivation_Level': 'High',
    'Internet_Access': 'Yes',
    'Family_Income': 'High',
    'Teacher_Quality': 'High',
    'School_Type': 'Public',
    'Peer_Influence': 'Positive',
    'Learning_Disabilities': 'No',
    'Parental_Education_Level': 'Postgraduate',
    'Distance_from_Home': 'Near',
    'Gender': 'Female'
}

student_average = {
    'Hours_Studied': 18,
    'Attendance': 75,
    'Sleep_Hours': 7,
    'Previous_Scores': 68,
    'Tutoring_Sessions': 1,
    'Physical_Activity': 3,
    'Parental_Involvement': 'Medium',
    'Access_to_Resources': 'Medium',
    'Extracurricular_Activities': 'No',
    'Motivation_Level': 'Medium',
    'Internet_Access': 'Yes',
    'Family_Income': 'Medium',
    'Teacher_Quality': 'Medium',
    'School_Type': 'Public',
    'Peer_Influence': 'Neutral',
    'Learning_Disabilities': 'No',
    'Parental_Education_Level': 'High School',
    'Distance_from_Home': 'Moderate',
    'Gender': 'Male'
}

student_at_risk = {
    'Hours_Studied': 6,
    'Attendance': 55,
    'Sleep_Hours': 5,
    'Previous_Scores': 52,
    'Tutoring_Sessions': 0,
    'Physical_Activity': 1,
    'Parental_Involvement': 'Low',
    'Access_to_Resources': 'Low',
    'Extracurricular_Activities': 'No',
    'Motivation_Level': 'Low',
    'Internet_Access': 'No',
    'Family_Income': 'Low',
    'Teacher_Quality': 'Low',
    'School_Type': 'Public',
    'Peer_Influence': 'Negative',
    'Learning_Disabilities': 'No',
    'Parental_Education_Level': 'High School',
    'Distance_from_Home': 'Far',
    'Gender': 'Male'
}

# Run Predictions
pred_high = predict_student_performance(student_high)
pred_avg = predict_student_performance(student_average)
pred_risk = predict_student_performance(student_at_risk)

print("=" * 60)
print("       STUDENT PERFORMANCE PREDICTION SYSTEM OUTPUT")
print("=" * 60)
print(f"Profile 1 (High Engagement)   -> Predicted Exam Score: {pred_high} / 100")
print(f"Profile 2 (Average Profile)   -> Predicted Exam Score: {pred_avg} / 100")
print(f"Profile 3 (At-Risk Profile)   -> Predicted Exam Score: {pred_risk} / 100")
print("=" * 60)

       STUDENT PERFORMANCE PREDICTION SYSTEM OUTPUT
Profile 1 (High Engagement)   -> Predicted Exam Score: 79.61 / 100
Profile 2 (Average Profile)   -> Predicted Exam Score: 63.97 / 100
Profile 3 (At-Risk Profile)   -> Predicted Exam Score: 49.55 / 100


In [ ]:
predict_code = '''import os
import joblib
import pandas as pd
import numpy as np

def load_inference_artifacts(model_path=None, prep_path=None):
    base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    if model_path is None:
        model_path = os.path.join(base_dir, "model", "best_student_model.joblib")
    if prep_path is None:
        prep_path = os.path.join(base_dir, "model", "preprocessor.joblib")

    if not os.path.exists(model_path) or not os.path.exists(prep_path):
        raise FileNotFoundError("Model or preprocessor artifact missing in 'model/' directory.")

    model = joblib.load(model_path)
    prep = joblib.load(prep_path)
    return model, prep

def predict_score(student_features: dict) -> float:
    """
    Accepts student attributes dictionary and returns predicted Exam_Score [0-100].
    """
    model, prep = load_inference_artifacts()
    df_input = pd.DataFrame([student_features])
    features_scaled = prep.transform(df_input)
    prediction = model.predict(features_scaled)[0]
    return round(float(np.clip(prediction, 0, 100)), 2)

if __name__ == "__main__":
    sample_student = {
        'Hours_Studied': 20,
        'Attendance': 85,
        'Sleep_Hours': 7,
        'Previous_Scores': 75,
        'Tutoring_Sessions': 1,
        'Physical_Activity': 3,
        'Parental_Involvement': 'High',
        'Access_to_Resources': 'High',
        'Extracurricular_Activities': 'Yes',
        'Motivation_Level': 'Medium',
        'Internet_Access': 'Yes',
        'Family_Income': 'Medium',
        'Teacher_Quality': 'Medium',
        'School_Type': 'Public',
        'Peer_Influence': 'Neutral',
        'Learning_Disabilities': 'No',
        'Parental_Education_Level': 'College',
        'Distance_from_Home': 'Near',
        'Gender': 'Female'
    }
    pred = predict_score(sample_student)
    print(f"Sample Student Prediction: {pred} marks")
'''

script_path = os.path.join(src_dir, "predict.py")
with open(script_path, "w") as f:
    f.write(predict_code)

print(f"✓ Standalone prediction script written to: {script_path}")

✓ Standalone prediction script written to: ../src/predict.py


In [19]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression

# 1. Force directory structure creation
os.makedirs("model", exist_ok=True)
os.makedirs("src", exist_ok=True)

# 2. Locate and load dataset
candidate_paths = [
    "student_data_cleaned.csv",
    "StudentPerformanceFactors.csv",
    "student_data.csv",
    os.path.join("data", "student_data_cleaned.csv"),
    os.path.join("..", "data", "student_data_cleaned.csv")
]

data_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if not data_path:
    raise FileNotFoundError("Could not find CSV dataset. Please upload 'student_data_cleaned.csv' to Colab.")

df = pd.read_csv(data_path)

# Ensure data cleaning guarantees
for col in ['Teacher_Quality', 'Parental_Education_Level', 'Distance_from_Home']:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])
df.loc[df['Exam_Score'] > 100, 'Exam_Score'] = 100

# 3. Separate features & target
X = df.drop(columns=['Exam_Score'])
y = df['Exam_Score']

num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

# 4. Construct and fit ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
    ],
    remainder='drop'
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train_processed = preprocessor.fit_transform(X_train)

# 5. Fit model
model = LinearRegression()
model.fit(X_train_processed, y_train)

# 6. Save joblib artifacts
prep_path = os.path.join("model", "preprocessor.joblib")
model_path = os.path.join("model", "best_student_model.joblib")

joblib.dump(preprocessor, prep_path)
joblib.dump(model, model_path)

# 7. Generate src/predict.py
predict_script_content = '''import os
import joblib
import pandas as pd
import numpy as np

def get_artifacts():
    # Resolve project root dynamically
    current_dir = os.path.dirname(os.path.abspath(__file__))
    project_root = os.path.abspath(os.path.join(current_dir, ".."))

    # Check current directory and project root for model folder
    candidate_dirs = [
        os.path.join(project_root, "model"),
        os.path.join(current_dir, "model"),
        "model"
    ]

    m_path, p_path = None, None
    for d in candidate_dirs:
        m_test = os.path.join(d, "best_student_model.joblib")
        p_test = os.path.join(d, "preprocessor.joblib")
        if os.path.exists(m_test) and os.path.exists(p_test):
            m_path = m_test
            p_path = p_test
            break

    if not m_path:
        raise FileNotFoundError("Could not find 'best_student_model.joblib' and 'preprocessor.joblib' in 'model/' folder.")

    model = joblib.load(m_path)
    prep = joblib.load(p_path)
    return model, prep

def predict_score(student_features: dict) -> float:
    """
    Accepts raw student features dictionary and returns predicted Exam_Score bound [0, 100].
    """
    model, prep = get_artifacts()
    df_input = pd.DataFrame([student_features])
    features_transformed = prep.transform(df_input)
    prediction = model.predict(features_transformed)[0]
    return round(float(np.clip(prediction, 0, 100)), 2)

if __name__ == "__main__":
    sample_student = {
        'Hours_Studied': 20,
        'Attendance': 85,
        'Sleep_Hours': 7,
        'Previous_Scores': 75,
        'Tutoring_Sessions': 1,
        'Physical_Activity': 3,
        'Parental_Involvement': 'High',
        'Access_to_Resources': 'High',
        'Extracurricular_Activities': 'Yes',
        'Motivation_Level': 'Medium',
        'Internet_Access': 'Yes',
        'Family_Income': 'Medium',
        'Teacher_Quality': 'Medium',
        'School_Type': 'Public',
        'Peer_Influence': 'Neutral',
        'Learning_Disabilities': 'No',
        'Parental_Education_Level': 'College',
        'Distance_from_Home': 'Near',
        'Gender': 'Female'
    }
    result = predict_score(sample_student)
    print(f"Sample Student Predicted Score: {result} / 100")
'''

script_path = os.path.join("src", "predict.py")
with open(script_path, "w") as f:
    f.write(predict_script_content)

# 8. File system verification
print("=" * 60)
print("FILE CREATION VERIFICATION")
print("=" * 60)
for file_to_check in [prep_path, model_path, script_path]:
    if os.path.exists(file_to_check):
        size_kb = os.path.getsize(file_to_check) / 1024
        print(f"  ✓ CREATED: {file_to_check:<35} ({size_kb:.1f} KB)")
    else:
        print(f"  ✗ FAILED:  {file_to_check}")
print("=" * 60)

FILE CREATION VERIFICATION
  ✓ CREATED: model/preprocessor.joblib           (6.6 KB)
  ✓ CREATED: model/best_student_model.joblib     (1.0 KB)
  ✓ CREATED: src/predict.py                      (2.2 KB)
